# Fig. 3 — Mean-field phase diagram in the $(w^{\rm p}, N_g)$ plane

Mean-field quantities on a 1000 × 1000 grid of pump rate $w^{\rm p}$ × gain atoms $N_g$, at $N_c = 3000$ and $r = 0.5$ (**panel a**): stability of the non-lasing solution, the condition for a stationary lasing solution (`cond_chaos`), the pump cap and the characteristic pump rates $w_{\min}$, $w_{\max}$.

**Outputs** (`../data/P_Ng/`): `P_mf.npy`, `mf_unstable_normal.npy`, `mf_cond_chaos.npy`, `mf_cond_pump.npy`, `mf_wmin.npy`, `mf_wmax.npy`.

In [ ]:
using WignerSymbols
using LinearAlgebra
using NPZ

In [ ]:
Threads.nthreads()

In [ ]:
function dipole_me(F,Fp,m,p) # Dipole matrix element of the D1 line between |F,m> and |F',m+p> (Rb87, I = 3/2)
    
    J_g = 1/2
    J_e = 1/2
    I = 3/2

    sixj = Float64.(wigner6j(J_g, J_e, 1, Fp, F, I))
    cg = Float64.(clebschgordan(Fp, m+p, 1, -p, F, m))

    return (-1)^(Fp+J_g+1+I)*sqrt((2*Fp+1)*(2*J_g+1))*cg*sixj
end

## Parameters

In [ ]:
# Grid
num_Nl = 1000
num_O = 1000

# Rb87 level structure (B in G, frequencies in 2π·GHz)
B = 2.7
gamma = 2*pi*5.75*1e-3 # D1 linewidth
gamma_D2 = 2*pi*6.06*1e-3 # D2 linewidth
w_t = 2*pi*377107.463380
w_t_D2 = 2*pi*384230.48446
w_e = 2*pi*(6.834-3*0.7*1e-3*B)
w_s = w_t+2*pi*0.30544+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*2*0.23*1e-3*B
w_r = w_t-2*pi*0.50906+2*pi*4.271677-2*pi*0.7*1e-3*B+2*pi*0.23*1e-3*B
w_u = w_t_D2-2*pi*0.22985+2*pi*4.271677-2*pi*0.7*1e-3*B-2*pi*0.93*1e-3*B

# Number of cooling (N_atom_I = N_c) and gain (N_atom_II = N_g) atoms
N_atom_I = 3000
N_atom_II = logrange(10,1e4,num_Nl)
N_atom = N_atom_I .+ N_atom_II'

# Auxiliary cavity mode b
gb = 3*2*pi*1e-3
omega = 40*gamma_D2
w_b = w_u + 2*2pi
kappa_b = 20*2*pi*1e-3

# Main cavity mode a
kappa = 0.07*2*pi*1e-3
gr = 0.78*2*pi*1e-3
gs = gr*abs(dipole_me(2,2,-2,0)/dipole_me(1,1,-1,0))
r = 0.5 # Squeezing parameter
eps_cav = 1*1e-3*2*pi
w_cav = (1/3)*(w_e+4*w_r-w_s)+eps_cav

## Adiabatic elimination

Laser frequencies, rotating-frame detunings and effective couplings obtained after eliminating the excited levels $u$, $r$, $s$ and then the modes $b$ and $s$ (steps 1–7), and the effective pumping, decay and dephasing rates of the atoms (step 8).

In [ ]:
# Step 1: find convenient frequencies for the lasers

w_lr = (-w_cav+3*w_r-w_s).-sqrt.(4*gr^2*N_atom.+(w_cav-w_r)^2)
w_ls = -2*w_lr.+(4*w_r-w_s)
w_l = 0.5*(w_b.+w_lr.-w_ls.+w_u.+sqrt.(4*gb^2*N_atom_I.+(w_b-w_u)^2 .-omega^2))

# Step 2: move to rotating frame

delta_b = w_b .- w_l .- (w_ls.-w_lr)./2
delta_u = w_u .- w_l .- (w_ls.-w_lr)./2
delta_1 = w_e.-(w_ls.-w_lr)./2
delta_s = w_s.-w_ls
delta_r = w_r.-(w_ls.+w_lr)./2
delta_cav = w_cav.-(w_ls.+w_lr)./2

# Step 3: eliminate level u

w_0_b = -0.25.*(omega.^2)./delta_u
w_c_b = delta_b.-(N_atom_I*gb^2)./delta_u
chi_b = (gb^2)./delta_u
g = -omega.*gb./(2*delta_u)

# Step 4: find Rabi frequencies

omega_r = sqrt.((w_ls.-2*w_e.-w_lr)./(1 ./(w_lr.+w_ls.-2*w_r).-2*tanh.(r).^2 ./(w_ls.-w_s)))
omega_s = (gr/gs)*omega_r.*tanh.(r).*(delta_s./delta_r)

# Step 5: eliminate levels r and s

w_0 = delta_1.-0.25.*((omega_r.^2)./delta_r.-(omega_s.^2)./delta_s)
w_c = delta_cav.-(N_atom*gr^2)./delta_r
chi = (gr^2)./delta_r.-(gs^2)./delta_s
lambda_r = -omega_r.*gr./(2*delta_r)
lambda_s = -omega_s.*gs./(2*delta_s)
g_02 = (lambda_r.^2).-lambda_s.^2

alfa = omega_r./(2*delta_r)
beta = gr./delta_r
delta = omega_s./(2*delta_s)
xi = gs./delta_s
alfa_b = omega./(2*delta_u)
beta_b = gb./delta_u

dec_eff = gamma*dipole_me(1,1,-1,0)^2*alfa.^2
P_eff = gamma*dipole_me(2,2,-2,0)^2*delta.^2
dep_eff = (gamma*dipole_me(1,2,-1,-1)^2*delta.^2 .+ gamma*dipole_me(2,1,-2,+1)^2*alfa.^2)/4
dec_eff_b = gamma_D2*(5/12)*alfa_b.^2
dep_eff_b = gamma_D2*(1/10)*alfa_b.^2/4

# Step 6: Eliminate mode b

dec = (g.^2*kappa_b)./((w_c_b.-w_0_b).^2 .+ kappa_b^2/4)
delta_lamb = w_0_b .+ w_0 .+ N_atom_I.*(g.^2).*(w_c_b.-w_0_b)./((kappa_b^2/4).+(w_c_b.-w_0_b).^2) # coefficient of s's term

# Step 7: Eliminate mode s

dec_f = dec.*g_02./(dec.^2/4 .+ (delta_lamb./N_atom_I).^2)
delta_lamb_f = (g_02.*delta_lamb.*N_atom_I)./((delta_lamb.^2).+(N_atom_I.*dec/2).^2)

# Step 8: include pumping and dephasing

dec_T = dec_eff
dep_T = dep_eff

P = logrange(1e-6,1.798e-2,num_O)
Γ_c = (4*g_02./(kappa.+dec_f)) # Purcell decay of qubits

P_T = P.+P_eff

C = (2*N_atom_II'.*g_02)./(((P_T.+dec_T)/2 .+ 2*dep_T).*(dec_f.+kappa))./(1 .+(4*delta_lamb_f.^2)./((kappa.+dec_f).+(P_T.+dec_T.+4*dep_T)).^2)
n_q = (P_T.-dec_T)./(2*(kappa.+dec_f)).*N_atom_II'
control = P_T./(kappa.+dec_f).-((P_T.+dec_T)./(2*(kappa.+dec_f))).*(1 .+ 1 ./ C)

## Stability matrix

Jacobian of the mean-field equations around the non-lasing solution of the full model.

In [ ]:
function mf_normal_full(κ,κb,g_0,g_b,Δ,Nc,Nl,P_c,γ_c,ϕ_c,P_l,γ_l,ϕ_l,χ)

    M = zeros(ComplexF64,4,4)
    
    sz_c = (P_c-γ_c)/(P_c+γ_c)
    sz_l = (P_l-γ_l)/(P_l+γ_l)
    Γ_l = P_l+γ_l+4*ϕ_l
    Γ_c = P_c+γ_c+4*ϕ_c
    M_c = P_c/(P_c+γ_c)
    M_l = P_l/(P_l+γ_l)

    M[1,1]=-κ/2
    M[1,3]=-im*g_0*Nc
    M[1,4]=-im*g_0*Nl

    M[2,2]=-κb/2-im*Δ-im*Nc*χ*M_c
    M[2,3]=-im*g_b*Nc

    M[3,1]=im*sz_c*g_0
    M[3,2]=im*sz_c*g_b
    M[3,3]=-(Γ_c/2+im*Δ)

    M[4,1]=im*g_0*sz_l
    M[4,4]=-Γ_l/2

    return M
end

## Phase diagram

In [ ]:
eigs_comp = zeros(ComplexF64,num_O,num_Nl,4)

Threads.@threads for i in 1:num_O
    for j in 1:num_Nl
        eigs_comp[i,j,:] = eigvals(mf_normal_full(kappa,kappa_b,sqrt(g_02[j]),g[j],w_0_b[j],N_atom_I,N_atom_II[j],P_eff[j],dec_eff[j]+dec_eff_b[j],dep_eff[j]+dep_eff_b[j],P_T[i,j],dec_T[j],dep_T[j],chi_b[j]))
    end
end

PD_comp = sum((real(eigs_comp).>0);dims=3)[:,:,1] # Number of unstable eigenvalues of the non-lasing solution (full model)

In [ ]:
# Stationarity of the lasing solution
γt = (dec_T.+P_T)/2 .+ 2*dep_T
γp = dec_T.+P_T
ss = C.*(P_T.-dec_T)./(P_T.+dec_T)
K  = kappa .+ dec_f
A  = 2 .* γt .+ K
D2 = delta_lamb_f.^2
B  = 4 .* D2 .+ A.^2
Z = 2 .* ss .* γt .* (2 .* γt .+ 3 .* K) .* B .+ K .* (4 .* D2 .* (K .- 10 .* γt) .+ (K .- 2 .* γt) .* A.^2)
M_scaled = A.^4 .+ 4 .* A.^2 .* γp .* (γt .* (1 .+ ss) .+ K) .+ 4 .* D2 .* ((K .- 2 .* γt).^2 .+ 4 .* γt .* (ss .- 1) .* γp)
T1 = -8 .* (ss .- 1) .* γt .* K .* A.^3 .* (γp .+ A).^2 .* B
T2 = (γp .+ A) .* M_scaled .* Z
T3 = -γp .* Z.^2
cond_chaos = (γp ./ (16 .* A.^4)) .* (T1 .+ T2 .+ T3)

cond_pump = P.*ones(num_O,num_Nl).-(200*1e-6*(2pi)) # Pump cap

purc = 4*g_02./(kappa.+dec_f) # Purcell rate of the gain atoms
wmax = (N_atom_II'.*purc)

## Save

In [ ]:
data_dir = joinpath("..", "data", "P_Ng") # Relative to this notebook

npzwrite(joinpath(data_dir, "P_mf.npy"), collect(P))
npzwrite(joinpath(data_dir, "mf_unstable_normal.npy"), PD_comp)
npzwrite(joinpath(data_dir, "mf_cond_chaos.npy"), cond_chaos)
npzwrite(joinpath(data_dir, "mf_cond_pump.npy"), cond_pump)
npzwrite(joinpath(data_dir, "mf_wmax.npy"), wmax[1,:])
npzwrite(joinpath(data_dir, "mf_wmin.npy"), dec_T[1,:])